# Module 4 — Sanctions / Compliance Entity Screening (OFAC SDN list)

**Scenario**: A compliance team needs to screen a batch of counterparty/vendor
names from a client's procurement system against the US Treasury OFAC
Specially Designated Nationals (SDN) list, flag fuzzy matches (typos,
transliterations, formatting differences), and produce a triage list for
analyst review.

**This notebook**:
1. Pulls the real, current OFAC SDN list
2. Screens a small illustrative "vendor master data" sample (constructed for
   this exercise — see note below) against SDN entity names using fuzzy matching
3. Risk-scores and bands the matches for analyst triage
4. Writes a stakeholder summary to `reports/ofac_summary.md`

**Note on the candidate list**: the OFAC SDN data itself is real and pulled
live. The 8 "candidate vendor names" below are illustrative — some are
deliberately close variants of real SDN entries (to demonstrate true-positive
fuzzy matches), and some are unrelated company names (to demonstrate clean
"no hit" results). A production version would use a client's actual vendor
master data.

**Caveat**: fuzzy name matching produces *candidate* matches for human review —
it is not a sanctions determination. A name match alone (especially for common
words) does not mean the counterparty is sanctioned; analysts must also check
identifying details (address, DOB, ID numbers) on the full SDN record.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from src.data_connectors.ofac_sdn import fetch_sdn_list
from src.risk_framework.fuzzy_matching import screen_names
from src.risk_framework.risk_scoring import RiskScoringEngine

## 1. Pull the OFAC SDN list

In [2]:
sdn = fetch_sdn_list()
print(f"SDN list: {len(sdn):,} entries")
sdn["sdn_type"].value_counts(dropna=False)

SDN list: 19,066 entries


sdn_type
<NA>          9741
individual    7481
vessel        1499
aircraft       344
NaN              1
Name: count, dtype: int64

In [3]:
# Entities (companies, organisations) have a blank sdn_type; individuals/vessels/
# aircraft are explicitly typed. We screen against entity names here.
entity_names = sdn[sdn["sdn_type"].isna()]["sdn_name"].dropna()
print(f"{len(entity_names):,} entity names to screen against")

9,741 entity names to screen against


## 2. Illustrative vendor master data sample

See the note above — three of these are deliberate near-matches to real SDN
entries (typo, abbreviation, formatting difference), the rest are unrelated
company names that should produce no match.

In [4]:
vendor_master_data = [
    "Anglo Caribbean Co Ltd",        # near-match: 'ANGLO-CARIBBEAN CO., LTD.'
    "Banco Nacional de Cuba SA",     # near-match: 'BANCO NACIONAL DE CUBA'
    "Aerocaribean Airlins",          # typo of 'AEROCARIBBEAN AIRLINES'
    "Acme Logistics Group Inc",
    "Global Freight Solutions Ltd",
    "Northwind Pharma Supplies BV",
    "Meridian Clinical Research LLC",
    "Sunrise Medical Devices Pty Ltd",
]

## 3. Fuzzy screening

In [5]:
screened = screen_names(vendor_master_data, entity_names, score_threshold=75, limit=1)
screened

,candidate_name,matched_name,score
0,Anglo Caribbean Co Ltd,"ANGLO-CARIBBEAN CO., LTD.",89.361702
1,Banco Nacional de Cuba SA,BANCO NACIONAL DE CUBA,93.617021
2,Aerocaribean Airlins,AEROCARIBBEAN AIRLINES,95.238095
3,Acme Logistics Group Inc,None,0.000000
4,Global Freight Solutions Ltd,None,0.000000
5,Northwind Pharma Supplies BV,None,0.000000
6,Meridian Clinical Research LLC,None,0.000000
7,Sunrise Medical Devices Pty Ltd,None,0.000000


## 4. Risk scoring and analyst triage bands

Match score (0-100) becomes the single risk factor (normalised 0-1). Bands
give analysts a clear triage order: Critical/High matches reviewed first,
Low (no qualifying match) recorded as "screened, cleared".

In [6]:
screened["match_strength"] = screened["score"] / 100
engine = RiskScoringEngine(weights={"match_strength": 1.0})
triaged = engine.score_dataframe(screened).sort_values("risk_score", ascending=False)
triaged[["candidate_name", "matched_name", "score", "risk_band"]]

,candidate_name,matched_name,score,risk_band
2,Aerocaribean Airlins,AEROCARIBBEAN AIRLINES,95.238095,Critical
1,Banco Nacional de Cuba SA,BANCO NACIONAL DE CUBA,93.617021,Critical
0,Anglo Caribbean Co Ltd,"ANGLO-CARIBBEAN CO., LTD.",89.361702,Critical
3,Acme Logistics Group Inc,None,0.000000,Low
4,Global Freight Solutions Ltd,None,0.000000,Low
5,Northwind Pharma Supplies BV,None,0.000000,Low
6,Meridian Clinical Research LLC,None,0.000000,Low
7,Sunrise Medical Devices Pty Ltd,None,0.000000,Low


## 5. Save processed outputs

Persist the screened/triaged candidate list to `data/processed/ofac/` for
downstream use (e.g. the Streamlit dashboard), alongside a metadata sidecar.

In [7]:
import json
from datetime import datetime, timezone
from pathlib import Path

processed_dir = Path("../data/processed/ofac")
processed_dir.mkdir(parents=True, exist_ok=True)

triaged.to_csv(processed_dir / "screening_results.csv", index=False)

with open(processed_dir / "processed.meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "sdn_entity_names_screened_against": len(entity_names),
        "candidates_screened": len(vendor_master_data),
        "score_threshold": 75,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "files": {
            "screening_results.csv": f"{len(triaged)} candidate names, fuzzy-matched and risk-banded",
        },
    }, f, indent=2)

print(f"Wrote {len(triaged)} screening rows to {processed_dir.resolve()}")

Wrote 8 screening rows to C:\Users\allif\Downloads\Risk Modelling\data\processed\ofac


## 6. Write stakeholder summary

In [8]:
from datetime import datetime, timezone
from pathlib import Path

hits = triaged[triaged["matched_name"].notna()]
cleared = triaged[triaged["matched_name"].isna()]

lines = []
lines.append("# Sanctions Screening Triage — OFAC SDN List\n")
lines.append(f"_Generated {datetime.now(timezone.utc).date()}. Screened {len(vendor_master_data)} "
              f"vendor names against {len(entity_names):,} OFAC SDN entity records "
              f"(downloaded live from Treasury)._\n")
lines.append("## What this means\n")
lines.append(f"{len(hits)} of {len(vendor_master_data)} screened names produced a fuzzy match "
              f"(score >= 75) against the SDN list and require analyst review before onboarding "
              f"or payment. {len(cleared)} produced no qualifying match and can be cleared, "
              f"pending standard due diligence.\n")
lines.append("## Matches requiring review\n")
lines.append("| Vendor name (candidate) | Closest SDN match | Score | Risk band |")
lines.append("|---|---|---|---|")
for _, r in hits.iterrows():
    lines.append(f"| {r['candidate_name']} | {r['matched_name']} | {r['score']:.0f} | {r['risk_band']} |")
lines.append("")
lines.append("## Cleared (no qualifying match)\n")
for _, r in cleared.iterrows():
    lines.append(f"- {r['candidate_name']}")
lines.append("")
lines.append("## Important caveats\n")
lines.append("- A fuzzy name match is a **lead for review**, not a sanctions determination. "
              "Analysts must compare addresses, dates of birth, ID numbers, and program tags on "
              "the full SDN record before escalating.\n")
lines.append("- Common/generic words (e.g. 'Holdings', 'Group', 'International') can produce "
              "false-positive matches purely from shared vocabulary — review the *whole* name, "
              "not just the score.\n")
lines.append("- The SDN list changes frequently; screening results are only valid as of the "
              "pull date above and should be refreshed before each screening run.\n")
lines.append("## Recommended next steps\n")
lines.append("1. Compliance analyst to review each 'matches requiring review' row against the "
              "full SDN record (address, aliases, program).\n")
lines.append("2. For confirmed matches, escalate per the firm's sanctions escalation procedure "
              "and freeze any pending transactions.\n")
lines.append("3. Re-run this screening on a recurring schedule (e.g. before each new vendor "
              "onboarding and periodically against the existing vendor base, since the SDN list "
              "changes over time).\n")

report_path = Path("../reports/ofac_summary.md")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {report_path.resolve()}")

Wrote C:\Users\allif\Downloads\Risk Modelling\reports\ofac_summary.md
